In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table
from utils.constants import PRED_COLS

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "num_rows",
    "num_parse_errors",
    "num_gen_errors",
    #*PRED_COLS
]

def error_stats_table(jobs, cols):

    print(f"Processing {len(jobs)} jobs.")
    
    table = {col_name: [] for col_name in cols}
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")
        num_rows = len(df)

        # LLM failed to produce proper json
        mask_no_errors = (df[PRED_COLS] != "\"PARSE ERROR\"").all(axis=1)
        df = df[mask_no_errors]
        num_parse_errors = list(mask_no_errors).count(False)

        # LLM did not follow instructions
        mask_generate_fail = df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        df = df[mask_generate_fail]
        num_gen_errors = list(mask_generate_fail).count(False)

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append error counts
        table["num_rows"].append(num_rows)
        table["num_parse_errors"].append(num_parse_errors)
        table["num_gen_errors"].append(num_gen_errors)

    return table

In [4]:
def aggregate_results(df, by_cols, cols, funs=None):
    grouped = df.groupby(
        by=by_cols,
        as_index=True,
    )
    
    if funs is None:
        funs = [
        "mean",
        "std",
        "count"
    ]

    agg = grouped.agg({col: funs for col in cols})

    return agg

In [5]:
def bold_extreme_values(s):
    # Bold max for mean
    if "mean" in s.name or "sum" in s.name:
        is_max = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_max]
    if "std" in s.name:
        is_min = s == s.min()
        return ['font-weight: bold' if v else '' for v in is_min]

    return ['' for v in s]

In [6]:
# Collect raw data
basepath = "./outputs/v2"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

res = pd.DataFrame(error_stats_table(jobs_list, cols))
res = prettify_table(res)

Processing 495 jobs.


In [7]:
# Show all columns
pd.options.display.max_columns = None
# Set float formatting
pd.options.display.float_format = '{:,.3f}'.format
bycols = [
    "model",
    "number_of_demonstrations",
    "type_of_demonstrations",
    "use_instructions"
]

use_models=[
    "Llama-3.1-8B-Instruct",
    "Llama-3.3-70B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Mistral-Small-3.2-24B-Instruct-2506",
    "Qwen3-VL-8B-Instruct",
    "Qwen2.5-72B-Instruct"
]

use_models = slice(None)

agg = aggregate_results(res, bycols, cols[5:], funs=["sum", "mean"])
# Drop count
agg = agg.loc[pd.IndexSlice[use_models, :, :, :], (slice(None), ["sum", "mean"])]
# Highlight max values for each column
agg.style.apply(bold_extreme_values, axis=0)